# Test local `data/amenities` and upload to Hugging Face

1. **Sanity-check** CSVs under `data/amenities/` (exist, non-empty, row counts).
2. **Upload** that folder to the dataset repo **`PropertyLens/Resealeflats`** at path **`amenities/`** (files become `amenities/mrt_stations.csv`, etc.).
3. **Optional:** download back from the Hub and compare checksums.

**Requirements**
- Repo-root **`.env`** with **`HF_TOKEN`** — a Hugging Face token that has **write** access to `PropertyLens/Resealeflats`.
- Run from `notebooks/` (or adjust `REPO_ROOT` logic if you open the notebook elsewhere).

> Clones can pull these files with `snapshot_download(..., allow_patterns=["amenities/**"])` or by path.

In [1]:
%pip install -q python-dotenv huggingface-hub pandas

Note: you may need to restart the kernel to use updated packages.


## 1. Paths and expected files

In [2]:
import hashlib
from pathlib import Path

_cwd = Path.cwd().resolve()
REPO_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
AMENITIES = REPO_ROOT / "data" / "amenities"

EXPECTED = (
    "mrt_stations.csv",
    "hawker_centres.csv",
    "schools.csv",
    "malls.csv",
)

print("REPO_ROOT:", REPO_ROOT)
print("AMENITIES:", AMENITIES)
for name in EXPECTED:
    p = AMENITIES / name
    print(f"  {name}: {'OK' if p.is_file() else 'MISSING'} ({p.stat().st_size if p.is_file() else 0} bytes)")

REPO_ROOT: /Users/bhuvesh/Documents/PropertyLens
AMENITIES: /Users/bhuvesh/Documents/PropertyLens/data/amenities
  mrt_stations.csv: OK (19053 bytes)
  hawker_centres.csv: OK (15100 bytes)
  schools.csv: OK (23099 bytes)
  malls.csv: OK (4643 bytes)


## 2. Test: load CSVs and row counts

In [3]:
import pandas as pd

assert AMENITIES.is_dir(), f"Create {AMENITIES} or run 00_download_amenity_data.ipynb"

rows = {}
for name in EXPECTED:
    p = AMENITIES / name
    assert p.is_file(), f"Missing {p}"
    df = pd.read_csv(p)
    assert len(df) > 0, f"{name} is empty"
    rows[name] = len(df)
    print(f"{name}: {len(df)} rows, columns: {list(df.columns)[:6]}{'...' if len(df.columns) > 6 else ''}")

print("\nAll amenity CSV tests passed.")

mrt_stations.csv: 161 rows, columns: ['name', 'lat', 'lng', 'type', 'address', 'postal_code']
hawker_centres.csv: 127 rows, columns: ['name', 'lat', 'lng', 'address']
schools.csv: 179 rows, columns: ['name', 'lat', 'lng', 'address', 'postal_code', 'type']
malls.csv: 42 rows, columns: ['name', 'lat', 'lng', 'address']

All amenity CSV tests passed.


## 3. Upload `data/amenities` → `PropertyLens/Resealeflats` path `amenities/`

Uses `HfApi.upload_folder`. First commit may create the `amenities/` tree on the dataset repo.

In [4]:
import os
from dotenv import load_dotenv
from huggingface_hub import HfApi

load_dotenv(REPO_ROOT / ".env")
token = os.getenv("HF_TOKEN", "").strip()
if not token:
    raise ValueError("Set HF_TOKEN in repo-root .env (token needs write access to the dataset).")

REPO_ID = "PropertyLens/Resealeflats"
PATH_IN_REPO = "amenities"

api = HfApi(token=token)
info = api.upload_folder(
    repo_id=REPO_ID,
    repo_type="dataset",
    folder_path=str(AMENITIES),
    path_in_repo=PATH_IN_REPO,
    commit_message="Add/update amenities CSVs (MRT, hawker, schools, malls)",
)
print("Commit:", getattr(info, "commit_url", None) or info)
print(f"Uploaded local {AMENITIES} -> {REPO_ID} ({PATH_IN_REPO}/)")

/Users/bhuvesh/Documents/PropertyLens/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Commit: https://huggingface.co/datasets/PropertyLens/Resealeflats/commit/133edc52cdb74704de51b0cafe9d17b476d26d81
Uploaded local /Users/bhuvesh/Documents/PropertyLens/data/amenities -> PropertyLens/Resealeflats (amenities/)


## 4. Optional round-trip test (download vs local SHA256)

In [5]:
from huggingface_hub import hf_hub_download

def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    h.update(path.read_bytes())
    return h.hexdigest()

for name in EXPECTED:
    local = AMENITIES / name
    remote_path = f"{PATH_IN_REPO}/{name}"
    dl = Path(
        hf_hub_download(
            repo_id=REPO_ID,
            repo_type="dataset",
            filename=remote_path,
            token=token,
        )
    )
    same = file_sha256(local) == file_sha256(dl)
    print(f"{name}: SHA256 match = {same}")

print("Round-trip check done.")

mrt_stations.csv: SHA256 match = True
hawker_centres.csv: SHA256 match = True
schools.csv: SHA256 match = True
malls.csv: SHA256 match = True
Round-trip check done.
